In [ ]:
!pip show langchain

Name: langchain
Version: 1.3.18
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.13/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 


In [ ]:
!pip install -qU langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.7/163.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.0/572.0 kB 19.9 MB/s eta 0:00:00


In [ ]:
!pip install -qU langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 18.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


In [ ]:
from langchain.chat_models import init_chat_model
from google.colab import userdata

In [ ]:
gemini_api_key = userdata.get('gemini_api_key')

In [ ]:
model = init_chat_model(
    model = "google_genai:gemini-3.5-flash",
    api_key = gemini_api_key,
)

Create skill demand tool

In [ ]:
!pip install -qU langchain-tavily

In [ ]:
from langchain_tavily import TavilySearch
from pprint import pprint

In [ ]:
tavily_api_key = userdata.get('TAVILY_API_KEY')

In [ ]:
skill_demand_tool = TavilySearch(
   max_results = 5,
   topic = "general",
   search_depth = "advanced",
   tavily_api_key = tavily_api_key
)

Create job Search Tool

In [ ]:
rapid_api_key = userdata.get('RAPID_API_KEY')

In [ ]:
import requests
from langchain.tools import tool
from google.colab import userdata

@tool
def search_jobs(skill: str, location: str) -> list:
    """Search for jobs requiring a specific skill using JSearch API from RapidAPI."""
    print(f"\nCalling search_jobs tool")
    print(f"Searching jobs for: {skill} in {location}")

    rapidapi_key = userdata.get('RAPID_API_KEY')

    url = "https://jsearch.p.rapidapi.com/search"
    headers = {
        "x-rapidapi-key": rapidapi_key,
        "x-rapidapi-host": "jsearch.p.rapidapi.com"
    }
    querystring = {
        "query": f"{skill} in {location}",
        "page": "1",
        "country": "in",
        "employment_types": "INTERN,FULLTIME",
        "job_requirements": "no_experience,under_3_years_experience"
    }

    response = requests.get(url, headers=headers, params=querystring)
    data = response.json()

    jobs = data.get("data", [])
    print(f"Found {len(jobs)} jobs\n")

    result = []
    for job in jobs:
        result.append({
            "title": job.get("job_title"),
            "company": job.get("employer_name"),
            "location": job.get("job_city"),
            "apply_link": job.get("job_apply_link")
        })
    return result

Create an Agent

In [ ]:
from langchain.agents import create_agent

In [ ]:
system_prompt = """You are a Skill-to-Career Mapping assistant that helps students understand skill demand and find matching job opportunities.

You have access to these tools:
- skill_demand_tool: Search for industry demand, salary insights, and career trends
- search_jobs: Find actual job listings requiring specific skills

Help the student by researching the skill they ask about and finding relevant opportunities.

Present results in a clean, readable format with clear sections and proper spacing. Include all job details with apply links. Don't use markdown format."""

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
checkpointer = InMemorySaver()

In [ ]:
agent = create_agent(
    model = model,
    tools = [skill_demand_tool, search_jobs],
    system_prompt = system_prompt,
    checkpointer=checkpointer,
    debug = True
)

In [ ]:
config = {"configurable": {
    "thread_id": "1"
    }}

In [ ]:
user_query = "What's the demand for generative AI in the industry and show me related job openings in India"

In [ ]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": user_query
        }
    ]
}, config = config)

[values] {'messages': [HumanMessage(content="What's the demand for generative AI in the industry and show me related job openings in India", additional_kwargs={}, response_metadata={}, id='deb76acf-5e86-4fed-a57f-13f8ca2e9ae3'), HumanMessage(content="What's the demand for generative AI in the industry and show me related job openings in India", additional_kwargs={}, response_metadata={}, id='8e21341d-bd64-463d-a7a6-097e9fb14b72')]}
[updates] {'model': {'messages': [AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_jobs', 'arguments': '{"skill": "generative AI", "location": "India"}'}, '__gemini_function_call_thought_signatures__': {'call_384791': 'Eu0OCuoOAWkUfRMgjztFsHLBx/LyhWRTVOlNzp7xx2J2EFGNsRJok6j68BFkxCDjn1/HRy2kdISMfaCcKJFPFdEOKanI1JAv2FZ68VFb8FEUV4mH6aRGscLPCexHbaOpbRZ1T31D1EX9hLTIbYRcN/rzInpWYcPTmHuRixQFOEDOBKbXaW1t9MJ8aDezHdi4PWdeZ1MwOgm7CdU306NQ/ugcBDOlG7WdC6Pe3RMAJXamr02ptqu+hqvF6LhCifdHQHAgj5tQrvBcBicq2uYlRnipG1/KGbkeIQP46lSwGYMceS+cmehmPjpaLcY647r

In [ ]:
print(response["messages"][-1].content[0]['text'])

DEMAND FOR GENERATIVE AI IN THE INDUSTRY

The demand for Generative AI (GenAI) professionals has undergone explosive growth globally and is accelerating rapidly through 2025 as companies transition from experimental pilots to full-scale enterprise adoption.


1. Massive Growth in Market and Postings
Global job listings citing generative AI skills have more than tripled in the last two years. In India, the artificial intelligence market is projected to reach approximately 17 billion USD by 2027. 

2. Critical Talent Shortage in India
According to major industry studies and national business reports, India faces an unprecedented gap in skilled AI talent. It is estimated that India will require at least 1 million skilled AI and machine learning professionals by 2026. This makes GenAI skills incredibly lucrative, as organizations are willing to pay premium salaries to secure trained engineers.

3. Competitive Salary Trends
Salary guides indicate that AI engineering compensation is projecte